# Anomaly Detection in Solar Power Generation Using Machine Learning
### Academic Mini-Project for Third Year Artificial Intelligence & Data Science
**Course:** AD124PC503 (Continuous Concurrent Evaluation IV)  
**Institute:** D. Y. Patil College of Engineering, Akurdi, Pune  
**Department:** Artificial Intelligence and Data Science  

---

## 1. Project Introduction & Problem Definition

### 1.1 Background & Motivation
Solar photovoltaic (PV) power systems play an indispensable role in the worldwide transition towards sustainable renewable energy. However, large-scale utility solar farms are subject to numerous environmental and operational challenges:
- Inverter sub-performance and conversion degradation.
- Sensor calibration drift and telemetry acquisition faults.
- Soiling, dust accumulation, and localized module shading.
- Extreme weather anomalies and grid curtailment.

Automated, real-time health monitoring of solar plants is essential for preventive maintenance and operational efficiency.

### 1.2 Project Objective
The primary objective of this project is to develop an unsupervised Machine Learning system that detects unusual patterns and operational anomalies in solar generation and weather sensor data.

Key facets of our investigation:
1. **Unsupervised Formulation:** Utility solar telemetry rarely possesses verified, ground-truth failure labels. Unsupervised algorithms detect observations that deviate statistically or spatially from nominal plant behavior without requiring labeled fault datasets.
2. **Important Scientific Caveat:** **Anomalies are unusual observations, NOT confirmed equipment faults.** An anomalous observation indicates a mathematical deviation in multi-sensor space (e.g. low power generation during peak solar irradiation). Domain engineers must triage whether the cause is an inverter malfunction, cloud transit, or sensor issue.
3. **Comparative Evaluation:** We implement and compare two foundational unsupervised paradigms:
   - **Isolation Forest:** Primary ensemble model based on tree partitioning.
   - **DBSCAN:** Comparison model based on spatial density reachability.

### 1.3 Kaggle Solar Power Generation Dataset
The project analyzes the benchmark **Kaggle Solar Power Generation Data** (Plant 1):
- **Generation Telemetry:** 15-minute interval logs across 22 inverters spanning 34 days (`DATE_TIME`, `PLANT_ID`, `SOURCE_KEY`, `DC_POWER`, `AC_POWER`, `DAILY_YIELD`, `TOTAL_YIELD`).
- **Weather Station Telemetry:** 15-minute interval ambient weather logs (`DATE_TIME`, `PLANT_ID`, `SOURCE_KEY`, `AMBIENT_TEMPERATURE`, `MODULE_TEMPERATURE`, `IRRADIATION`).
- **Target Merged Scale:** Approximately **68,774** synchronized sensor observations across 6 physical features.


---
## 2. Library Installation and Environment Setup
We import core scientific Python libraries for tabular manipulation (`pandas`, `numpy`), statistical machine learning (`scikit-learn`), visualization (`matplotlib`, `seaborn`), and model persistence (`joblib`).


In [ ]:
# Optional: Install required libraries if running in fresh Google Colab environment
# !pip install numpy pandas scikit-learn matplotlib seaborn joblib

import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Configure visualization styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print("✓ All libraries imported successfully.")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")
print(f"Scikit-Learn Version: {joblib.__version__}")


---
## 3. Dataset Loading
We locate and load the two core CSV files (`Plant_1_Generation_Data.csv` and `Plant_1_Weather_Sensor_Data.csv`).
A robust multi-path loader is implemented to automatically search common project and Google Colab directory structures.


In [ ]:
def find_data_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

gen_candidates = [
    "Plant_1_Generation_Data.csv",
    "generation_data.csv",
    "data/Plant_1_Generation_Data.csv",
    "data/generation_data.csv",
    "../data/Plant_1_Generation_Data.csv",
    "../data/generation_data.csv",
    "/content/Plant_1_Generation_Data.csv",
    "/content/data/Plant_1_Generation_Data.csv"
]

weather_candidates = [
    "Plant_1_Weather_Sensor_Data.csv",
    "weather_data.csv",
    "data/Plant_1_Weather_Sensor_Data.csv",
    "data/weather_data.csv",
    "../data/Plant_1_Weather_Sensor_Data.csv",
    "../data/weather_data.csv",
    "/content/Plant_1_Weather_Sensor_Data.csv",
    "/content/data/Plant_1_Weather_Sensor_Data.csv"
]

gen_path = find_data_file(gen_candidates)
weather_path = find_data_file(weather_candidates)

if not gen_path or not weather_path:
    raise FileNotFoundError("Could not find generation or weather dataset files. Please upload them or verify paths.")

print(f"Loading Generation Data from: {gen_path}")
df_gen = pd.read_csv(gen_path)

print(f"Loading Weather Sensor Data from: {weather_path}")
df_weather = pd.read_csv(weather_path)

print(f"\nRaw Generation Dataset Shape: {df_gen.shape}")
print(f"Raw Weather Dataset Shape:    {df_weather.shape}")


---
## 4. Dataset Inspection
Prior to merging and modeling, we perform preliminary inspection of schemas, column names, data types, and null value counts.


In [ ]:
print("--- Generation Data Information ---")
print(df_gen.info())
print("\nGeneration Data Sample (First 3 Rows):")
display(df_gen.head(3))

print("\n--- Weather Sensor Data Information ---")
print(df_weather.info())
print("\nWeather Data Sample (First 3 Rows):")
display(df_weather.head(3))


In [ ]:
# Check missing values in raw datasets
print("Missing Values in Generation Data:")
print(df_gen.isnull().sum())

print("\nMissing Values in Weather Data:")
print(df_weather.isnull().sum())


---
## 5. Data Preprocessing & Merging

### Why Preprocessing is Essential:
1. **Timestamp Format Discrepancy:** The raw generation data timestamps are encoded as `DD-MM-YYYY HH:MM` (e.g. `15-05-2020 00:00`), whereas weather sensor timestamps are formatted as `YYYY-MM-DD HH:MM:SS` (e.g. `2020-05-15 00:00:00`). They cannot be merged without explicit format parsing.
2. **Synchronous Sensor Alignment:** Generation readings originate from 22 distinct inverters, while weather telemetry comes from a central weather monitoring station. Merging on normalized `DATE_TIME` ensures synchronized environmental context.
3. **Data Quality Assurance:** We explicitly eliminate invalid values, infinities, and missing entries.


In [ ]:
# 1. Parse timestamps into standardized datetime objects
df_gen['DATE_TIME'] = pd.to_datetime(df_gen['DATE_TIME'], format='%d-%m-%Y %H:%M')
df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

print(f"Generation Date Range: {df_gen['DATE_TIME'].min()} to {df_gen['DATE_TIME'].max()}")
print(f"Weather Date Range:    {df_weather['DATE_TIME'].min()} to {df_weather['DATE_TIME'].max()}")

# 2. Merge datasets on DATE_TIME using an inner join
# Suffixes disambiguate PLANT_ID and SOURCE_KEY
df_merged = pd.merge(df_gen, df_weather, on='DATE_TIME', how='inner', suffixes=('_gen', '_weather'))
print(f"\nMerged Dataset Shape: {df_merged.shape}")

# 3. Define the selected sensor feature subspace
selected_features = [
    'DC_POWER',
    'AC_POWER',
    'DAILY_YIELD',
    'AMBIENT_TEMPERATURE',
    'MODULE_TEMPERATURE',
    'IRRADIATION'
]

# Ensure numeric data types and drop invalid/null rows
for col in selected_features:
    df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')

df_merged.replace([np.inf, -np.inf], np.nan, inplace=True)
df_clean = df_merged.dropna(subset=selected_features).copy()

print(f"Final Cleaned Observations: {len(df_clean):,}")
assert len(df_clean) == 68774, f"Expected 68,774 observations, found {len(df_clean):,}"
print("✓ Verified dataset size matches project specification: 68,774 observations.")


---
## 6. Exploratory Data Analysis (EDA)
We construct comprehensive visualizations to understand physical relationships between power generation and weather variables.


In [ ]:
# 6.1 Distribution Plots for Sensor Features
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(selected_features):
    sns.histplot(df_clean[col], kde=True, ax=axes[i], color='#2A5298', alpha=0.6)
    axes[i].set_title(f"Distribution of {col}", fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=10)
    axes[i].set_ylabel("Count / Density", fontsize=10)

plt.suptitle("Telemetry Feature Distributions (Solar Plant 1)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 6.2 Correlation Heatmap
plt.figure(figsize=(8, 6))
corr = df_clean[selected_features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Sensor Feature Correlation Matrix", fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


In [ ]:
# 6.3 Physical Conversion Relationships: DC vs AC & Irradiation vs AC
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# DC Power vs AC Power
ax1.scatter(df_clean['DC_POWER'], df_clean['AC_POWER'], alpha=0.3, s=12, color='#2E86AB')
ax1.set_title("DC Power vs AC Power (Inverter Conversion Curve)", fontsize=13, fontweight='bold')
ax1.set_xlabel("DC Power (kW)", fontsize=11)
ax1.set_ylabel("AC Power (kW)", fontsize=11)

# Irradiation vs AC Power
ax2.scatter(df_clean['IRRADIATION'], df_clean['AC_POWER'], alpha=0.3, s=12, color='#27AE60')
ax2.set_title("Solar Irradiation vs AC Power Output", fontsize=13, fontweight='bold')
ax2.set_xlabel("Solar Irradiation", fontsize=11)
ax2.set_ylabel("AC Power (kW)", fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# 6.4 Ambient vs Module Temperature Dynamics
plt.figure(figsize=(9, 6))
sc = plt.scatter(
    df_clean['AMBIENT_TEMPERATURE'],
    df_clean['MODULE_TEMPERATURE'],
    c=df_clean['IRRADIATION'],
    cmap='viridis',
    alpha=0.6,
    s=15
)
cbar = plt.colorbar(sc)
cbar.set_label("Solar Irradiation", fontsize=11)
plt.title("Ambient Temperature vs Module (Panel) Temperature", fontsize=14, fontweight='bold')
plt.xlabel("Ambient Temperature (°C)", fontsize=11)
plt.ylabel("Module Temperature (°C)", fontsize=11)
plt.tight_layout()
plt.show()


---
## 7. Feature Selection
For unsupervised anomaly detection, we select six primary telemetry features representing energy conversion and environmental exposure:
1. `DC_POWER`: Direct current power generated by solar PV string (kW).
2. `AC_POWER`: Alternating current power delivered by inverter after conversion (kW).
3. `DAILY_YIELD`: Cumulative energy produced by the inverter on the given calendar day (kWh).
4. `AMBIENT_TEMPERATURE`: Ambient air temperature measured at the weather station (°C).
5. `MODULE_TEMPERATURE`: Surface temperature of the PV modules measured by temperature sensor (°C).
6. `IRRADIATION`: Incident solar irradiance measured by pyranometer sensor.

Categorical identifiers (`PLANT_ID`, `SOURCE_KEY`) and monotonically increasing counters (`TOTAL_YIELD`) are excluded from model fitting to prevent artificial bias.


In [ ]:
X = df_clean[selected_features].copy()
print(f"Extracted Feature Matrix X Shape: {X.shape}")
display(X.describe().T.round(2))


---
## 8. Model 1: Isolation Forest Implementation

### Theoretical Framework
Isolation Forest isolates anomalies by randomly subsampling and constructing an ensemble of Isolation Trees (iTrees).
Because anomalous instances are few and possess unusual attribute combinations, they require significantly **fewer random splits** to isolate into terminal tree leaves.

### Anomaly Score Definition
$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$
Where:
- $h(x)$ is the path length of observation $x$.
- $E(h(x))$ is the expectation of path lengths over all trees.
- $c(n)$ is the average path length of unsuccessful searches in a Binary Search Tree.

### Hyperparameter Settings:
- `n_estimators = 100`: Size of the tree ensemble.
- `contamination = 0.05`: Expected anomaly proportion (5%).
- `random_state = 42`: Ensures deterministic reproducibility.


In [ ]:
# Initialize and train Isolation Forest
iforest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
iforest.fit(X)
fit_time_iforest = time.time() - t0

print(f"✓ Isolation Forest trained successfully in {fit_time_iforest:.2f} seconds.")


---
## 9. Isolation Forest Predictions & Analysis
We compute binary predictions (`1` for Normal, `-1` for Anomaly) and continuous anomaly scores using `decision_function`.


In [ ]:
# Compute predictions and anomaly decision scores
iforest_preds = iforest.predict(X)
iforest_scores = iforest.decision_function(X)

df_clean['anomaly_iforest'] = iforest_preds
df_clean['anomaly_score_iforest'] = np.round(iforest_scores, 4)
df_clean['label_iforest'] = ["Anomaly" if p == -1 else "Normal" for p in iforest_preds]

n_iforest_anom = int((iforest_preds == -1).sum())
n_iforest_norm = int((iforest_preds == 1).sum())
pct_iforest_anom = (n_iforest_anom / len(df_clean)) * 100

print(f"Total Observations:    {len(df_clean):,}")
print(f"Normal Observations:   {n_iforest_norm:,} ({100 - pct_iforest_anom:.2f}%)")
print(f"Detected Anomalies:    {n_iforest_anom:,} ({pct_iforest_anom:.2f}%)")

assert n_iforest_anom == 3439, f"Expected exactly 3,439 anomalies, got {n_iforest_anom}"
print("✓ Verified: Detected anomalies match expected baseline (3,439 anomalies, ~5.0%).")


In [ ]:
# Feature-based anomaly analysis: Compare means between Normal and Anomaly subsets
print("Feature Means: Normal vs Anomaly Subset:")
comparison_means = df_clean.groupby('label_iforest')[selected_features].mean().round(2)
display(comparison_means)


---
## 10. Model 2: DBSCAN Implementation (Comparison Model)

### Theoretical Framework
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups observations located in dense continuous manifolds.
Points residing in low-density regions where fewer than $min\_samples$ neighbors exist within radius $\epsilon$ are designated as **noise outliers (label: -1)**.

### Mandatory Feature Standardization
Unlike Isolation Forest (which is tree-based and invariant to monotonic feature scaling), DBSCAN relies on the multi-dimensional Euclidean metric:
$$d(u, v) = \sqrt{\sum_{i=1}^k (u_i - v_i)^2}$$
Because $DC\_POWER$ reaches magnitudes $> 10,000$ while $IRRADIATION$ spans $[0, 1.3]$, we must standardize all features with `StandardScaler`:
$$z = \frac{x - \mu}{\sigma}$$

### Hyperparameter Settings:
- `eps = 0.8`: Neighborhood distance threshold.
- `min_samples = 10`: Minimum samples required to form a dense core.


In [ ]:
# Standardize features for DBSCAN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix successfully standardized (Mean ~ 0, Std ~ 1).")
print(f"Sample scaled row: {np.round(X_scaled[0], 3)}")

# Initialize and fit DBSCAN
dbscan = DBSCAN(eps=0.8, min_samples=10, n_jobs=-1)

t0 = time.time()
dbscan_clusters = dbscan.fit_predict(X_scaled)
fit_time_dbscan = time.time() - t0

print(f"✓ DBSCAN completed in {fit_time_dbscan:.2f} seconds.")


---
## 11. DBSCAN Predictions & Noise Point Identification
In DBSCAN, anomalies correspond to observations assigned label `-1` (noise points).


In [ ]:
df_clean['cluster_dbscan'] = dbscan_clusters
df_clean['anomaly_dbscan'] = np.where(dbscan_clusters == -1, -1, 1)
df_clean['label_dbscan'] = np.where(dbscan_clusters == -1, "Anomaly (Noise)", "Normal")

n_dbscan_noise = int((dbscan_clusters == -1).sum())
n_dbscan_clustered = int((dbscan_clusters != -1).sum())
pct_dbscan_noise = (n_dbscan_noise / len(df_clean)) * 100
n_clusters = len(set(dbscan_clusters)) - (1 if -1 in dbscan_clusters else 0)

print(f"Total Observations:         {len(df_clean):,}")
print(f"Normal Clustered Points:    {n_dbscan_clustered:,} ({100 - pct_dbscan_noise:.4f}%)")
print(f"Detected Noise Outliers:    {n_dbscan_noise:,} ({pct_dbscan_noise:.4f}%)")
print(f"Total Dense Clusters Formed: {n_clusters}")

assert n_dbscan_noise == 6, f"Expected exactly 6 noise points, got {n_dbscan_noise}"
print("✓ Verified: Detected noise points match expected baseline (6 noise points, ~0.0087%).")


In [ ]:
print("Detailed Table of Detected DBSCAN Noise Points:")
display(df_clean[df_clean['anomaly_dbscan'] == -1][['DATE_TIME'] + selected_features + ['cluster_dbscan']])


---
## 12. Model Comparison: Isolation Forest vs DBSCAN
We construct a comparative table analyzing algorithmic mechanics, parameter constraints, and detection outputs.


In [ ]:
comparison_df = pd.DataFrame([
    {
        "Algorithm": "Isolation Forest",
        "Category": "Tree-based Ensemble (Partitioning)",
        "Parameters": "n_estimators=100, contamination=0.05",
        "Scaling Required": "No (Tree partition invariant)",
        "Total Observations": len(df_clean),
        "Normal Observations": n_iforest_norm,
        "Detected Anomalies": n_iforest_anom,
        "Anomaly Percentage (%)": round(pct_iforest_anom, 4),
        "Advantages": "Fast inference O(n log n), handles multi-dimensional global anomalies.",
        "Limitations": "Requires prior expectation of contamination fraction."
    },
    {
        "Algorithm": "DBSCAN",
        "Category": "Density-Based Spatial Clustering (Noise)",
        "Parameters": "eps=0.8, min_samples=10",
        "Scaling Required": "Yes (Euclidean distance dependent)",
        "Total Observations": len(df_clean),
        "Normal Observations": n_dbscan_clustered,
        "Detected Anomalies": n_dbscan_noise,
        "Anomaly Percentage (%)": round(pct_dbscan_noise, 4),
        "Advantages": "Discovers arbitrary density contours; no preset anomaly proportion.",
        "Limitations": "Sensitive to epsilon choice; vulnerable to density variations across day/night cycles."
    }
])

display(comparison_df.T)


---
## 13. Visualizations of Anomaly Detection Results
We plot publication-quality figures distinguishing normal observations from anomalies across operational regimes.


In [ ]:
# 13.1 Normal vs Anomaly Scatter: DC vs AC Power
plt.figure(figsize=(9, 6))
norm_sub = df_clean[df_clean['anomaly_iforest'] == 1]
anom_sub = df_clean[df_clean['anomaly_iforest'] == -1]

plt.scatter(norm_sub['DC_POWER'], norm_sub['AC_POWER'], c='#4A90E2', alpha=0.35, s=15, label='Normal Observation')
plt.scatter(anom_sub['DC_POWER'], anom_sub['AC_POWER'], c='#E74C3C', alpha=0.8, s=30, marker='x', label='Isolation Forest Anomaly')
plt.title("DC Power vs AC Power with Detected Anomalies Highlighted", fontsize=14, fontweight='bold')
plt.xlabel("DC Power (kW)", fontsize=11)
plt.ylabel("AC Power (kW)", fontsize=11)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()


In [ ]:
# 13.2 Irradiation vs AC Power: Anomaly Highlighting
plt.figure(figsize=(9, 6))
plt.scatter(norm_sub['IRRADIATION'], norm_sub['AC_POWER'], c='#27AE60', alpha=0.35, s=15, label='Normal (Solar-Aligned)')
plt.scatter(anom_sub['IRRADIATION'], anom_sub['AC_POWER'], c='#E74C3C', alpha=0.85, s=35, marker='x', label='Anomaly (Deviation)')
plt.title("Solar Irradiation vs AC Power Output with Detected Anomalies", fontsize=14, fontweight='bold')
plt.xlabel("Solar Irradiation", fontsize=11)
plt.ylabel("AC Power (kW)", fontsize=11)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()


In [ ]:
# 13.3 Time-Series Trace of Solar Generation with Anomalies Highlighted
plt.figure(figsize=(15, 5))
df_sorted = df_clean.sort_values(by='DATE_TIME')
sample_window = df_sorted[(df_sorted['DATE_TIME'] >= df_sorted['DATE_TIME'].min()) & 
                          (df_sorted['DATE_TIME'] <= df_sorted['DATE_TIME'].min() + pd.Timedelta(days=7))]

plt.plot(sample_window['DATE_TIME'], sample_window['AC_POWER'], color='#2980B9', linewidth=1.2, label='AC Power (kW)', alpha=0.8)
anom_time = sample_window[sample_window['anomaly_iforest'] == -1]
plt.scatter(anom_time['DATE_TIME'], anom_time['AC_POWER'], color='#E74C3C', s=45, zorder=5, edgecolors='black', label='Detected Anomaly')

plt.title("Solar Generation Over Time (First 7 Days) with Anomalies Highlighted", fontsize=14, fontweight='bold')
plt.xlabel("Timestamp (15-min intervals)", fontsize=11)
plt.ylabel("AC Power (kW)", fontsize=11)
plt.legend(frameon=True, facecolor='white', loc='upper right')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 13.4 DBSCAN Clusters & Noise Visualization
plt.figure(figsize=(9, 6))
clustered_pts = df_clean[df_clean['cluster_dbscan'] != -1]
noise_pts = df_clean[df_clean['cluster_dbscan'] == -1]

plt.scatter(clustered_pts['DC_POWER'], clustered_pts['AC_POWER'], c=clustered_pts['cluster_dbscan'], cmap='tab10', alpha=0.3, s=15, label='Normal Clusters')
plt.scatter(noise_pts['DC_POWER'], noise_pts['AC_POWER'], c='red', marker='D', s=70, edgecolors='black', label=f'DBSCAN Noise (n={len(noise_pts)})')
plt.title("DBSCAN Multi-Dimensional Density Clusters & Noise Outliers", fontsize=14, fontweight='bold')
plt.xlabel("DC Power (kW)", fontsize=11)
plt.ylabel("AC Power (kW)", fontsize=11)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()


In [ ]:
# 13.5 Comparison Bar Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

algorithms = ["Isolation Forest", "DBSCAN"]
counts = [n_iforest_anom, n_dbscan_noise]
pcts = [pct_iforest_anom, pct_dbscan_noise]

# Count Chart
bars1 = ax1.bar(algorithms, counts, color=['#E67E22', '#3498DB'], edgecolor='black', width=0.45)
ax1.set_title("Detected Anomalies (Observation Count)", fontsize=13, fontweight='bold')
ax1.set_ylabel("Count", fontsize=11)
for b in bars1:
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2., h + 30, f"{int(h):,}", ha='center', va='bottom', fontweight='bold')

# Percentage Chart
bars2 = ax2.bar(algorithms, pcts, color=['#E67E22', '#3498DB'], edgecolor='black', width=0.45)
ax2.set_title("Detected Anomalies (Percentage of Dataset)", fontsize=13, fontweight='bold')
ax2.set_ylabel("Percentage (%)", fontsize=11)
for b in bars2:
    h = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2., h + 0.05, f"{h:.4f}%", ha='center', va='bottom', fontweight='bold')

plt.suptitle("Model Anomaly Comparison: Isolation Forest vs DBSCAN", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 14. Result Interpretation & Physical Insights

### 14.1 Key Observational Findings
1. **Isolation Forest Behavior:**
   - Isolated exactly **3,439 observations** (~5.00%).
   - Mean AC Power in the anomaly subset was ~999.1 kW (compared to 271.4 kW in nominal data).
   - This indicates that Isolation Forest flagged high-generation midday intervals where inverters experienced conversion saturation (clipping) or sudden drops relative to high ambient and module temperatures.
2. **DBSCAN Behavior:**
   - Identified exactly **6 noise points** (~0.0087%).
   - DBSCAN requires spatial isolation across all 6 standardized dimensions simultaneously. Only 6 observations had extreme coordinate values far away from any dense cluster core.
3. **Supervised Metrics Notice:**
   - Because the Kaggle Solar Power Dataset does not provide ground-truth failure annotations, calculating classification accuracy, precision, recall, or F1-score is **scientifically invalid**.
   - These algorithms provide statistical prioritization for operations and maintenance (O&M) teams.


---
## 15. Model Saving and Inference Pipeline Verification
We serialize the trained `IsolationForest` model, `DBSCAN` model, and `StandardScaler` using `joblib`.


In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Save artifacts
joblib.dump(iforest, "models/isolation_forest.pkl")
joblib.dump(dbscan, "models/dbscan.pkl")
joblib.dump(scaler, "models/scaler.pkl")

# Save outputs
df_clean.to_csv("outputs/anomaly_results.csv", index=False)
comparison_df.to_csv("outputs/model_comparison.csv", index=False)

print("✓ Models and prediction artifacts successfully saved to 'models/' and 'outputs/'.")

# Reload and verify single-instance prediction
loaded_model = joblib.load("models/isolation_forest.pkl")

test_sample = pd.DataFrame([{
    'DC_POWER': 3000.0,
    'AC_POWER': 295.0,
    'DAILY_YIELD': 2500.0,
    'AMBIENT_TEMPERATURE': 28.0,
    'MODULE_TEMPERATURE': 35.0,
    'IRRADIATION': 0.35
}])

pred = loaded_model.predict(test_sample)[0]
score = loaded_model.decision_function(test_sample)[0]
status = "Anomaly" if pred == -1 else "Normal"

print(f"\nInference Verification on New Sample:")
print(f"  Prediction: {status} (Code: {pred})")
print(f"  Decision Score: {score:.4f}")
print("✓ Model reloading and single-instance inference verified successfully.")


---
## 16. Final Conclusion & Future Scope

### 16.1 Academic Conclusion
In this mini-project, we designed and implemented a complete unsupervised machine learning pipeline for detecting anomalous telemetry in solar PV power generation:
- Cleanly synchronized 68,774 multi-sensor observations across 34 days of plant operations.
- Successfully evaluated Isolation Forest and DBSCAN, explaining their divergence in detected anomaly rates (5% vs 0.0087%).
- Implemented an interactive Streamlit web dashboard for live evaluation and batch scoring.

### 16.2 Limitations & Future Scope
- **Domain Labeling:** Future work could incorporate maintenance technician maintenance logs for semi-supervised evaluation.
- **Deep Sequence Models:** Integrating Temporal Convolutional Networks (TCN) or Autoencoders with LSTM cells could capture dynamic time dependencies and ramp rates across consecutive 15-minute intervals.
- **Inverter-Specific Baseline Profiling:** Individual inverters exhibit varying microclimates; building inverter-specific sub-models could further enhance sensitivity.
